# Importing dataset

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix, roc_curve, auc

#Load sample dataset(Titanic as an example)
data = pd.read_csv('/kaggle/input/titanic/train.csv')

#Select and preprocess features
X = data[["Pclass","Sex","Age","Fare","SibSp","Parch"]]
X["Age"].fillna(X["Age"].median(), inplace=True)
X = pd.get_dummies(X, drop_first=True)
y = data["Survived"]

# Split dataset

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state =42,stratify=y)


# Ensemble Methods-Random Forest and Gradient Boosting

In [ ]:
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier

# Random Forest

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100,random_state=42)
rf_model.fit(X_train, y_train)
rf_acc = rf_model.score(X_test, y_test)
print(f"Random Forest Accuracy:{rf_acc:2f}")

# Gradient Boosting

In [ ]:
gb_model = GradientBoostingClassifier(n_estimators=100,random_state=42)
gb_model.fit(X_train, y_train)
gb_acc = gb_model.score(X_test, y_test)
print(f"Gradient Boosting Accuracy:{gb_acc:2f}")


# Handling Imbalancing

In [ ]:
#SMOTE-Synthetic Minority Overssampling Technique


In [ ]:
!pip install -U scikit-learn imbalanced-learn

In [ ]:
import os, sys
os.kill(os.getpid(),9)

In [ ]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:\n", y_train.value_counts())
print("\nAfter SMOTE:\n", y_train_res.value_counts())

#Retrain model on balanced data
rf_model.fit(X_train_res, y_train_res)
print(f"Random Forest(SMOTE) Accuracy: {rf_model.score(X_test,y_test):2f}")

# Feature Importance Analysis

In [ ]:
#After building our model, we can ask: "Which factors were most important for survival?"
#How It Works

    #Random Forest tracks which features most reduce uncertainty

     #Each feature gets an "importance score"

     #Sum of all scores = 1

#Expected Results (Titanic)
       #Sex (most important - "women and children first")

      #Passenger Class (Pclass - class divide)
    
     #Age (children prioritized)

    #Fare (correlates with class)

# Demo

In [ ]:
import matplotlib.pyplot as plt

# Feature importance from Random Forest
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances.sort_values().plot(kind="barh", figsize=(6,4), title="Feature Importance")
plt.show()


# Probability Calibration

In [ ]:
#A weather forecaster is "well-calibrated"
#if on the days they say "70% chance of rain," it actually rains about 70% of the time.
#If our model is poorly calibrated, when it predicts a 90% survival chance,
# maybe only 60% of those people actually survive. We can train a small "corrector" model (like Platt Scaling)
#to take the model's guesses and adjust them to be more accurate probabilities.

# Demo

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

# Calibrate Random Forest
calibrated_rf = CalibratedClassifierCV(rf_model, method='sigmoid', cv=3)
calibrated_rf.fit(X_train_res, y_train_res)

# Compare probabilities before and after calibration
probs_before = rf_model.predict_proba(X_test)[:, 1]
probs_after = calibrated_rf.predict_proba(X_test)[:, 1]

print(f"Before calibration (mean prob): {probs_before.mean():.3f}")
print(f"After calibration  (mean prob): {probs_after.mean():.3f}")


# Threshold Optimization

In [ ]:
#What if we want to be really, really sure before we predict 'survived'? We can change the threshold."

# Demo

In [ ]:
from sklearn.metrics import precision_recall_curve

# Get calibrated probabilities
y_scores = calibrated_rf.predict_proba(X_test)[:, 1]

# Compute precision-recall curve
precisions, recalls, thresholds = precision_recall_curve(y_test, y_scores)

# Find best threshold for max F1
f1_scores = 2 * (precisions * recalls) / (precisions + recalls)
best_thresh = thresholds[np.argmax(f1_scores)]
print(f"Optimal threshold: {best_thresh:.2f}")

# Apply optimized threshold
y_pred_opt = (y_scores >= best_thresh).astype(int)

print("\nClassification Report (optimized threshold):")
print(classification_report(y_test, y_pred_opt))


# For Clarification- SMOTE and Stratify

In [ ]:
#SMOTE and stratify both help deal with class imbalance, but they work at different stages and in different ways.


# STRATIFY

In [ ]:
#When you split your dataset into training and testing sets,
# you might accidentally create a test set with too few minority-class examples.
# Using stratify=y makes sure both sets have the same class ratio as the original data.

# Demo

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

# Let's simulate an imbalanced dataset
data = {
    "feature": range(1, 11),
    "target": [0,0,0,0,0,0,0,1,1,1]  # 70% zeros, 30% ones
}
df = pd.DataFrame(data)
X = df[["feature"]]
y = df["target"]

# Split WITHOUT stratify
X_train1, X_test1, y_train1, y_test1 = train_test_split(X, y, test_size=0.3, random_state=42)

# Split WITH stratify
X_train2, X_test2, y_train2, y_test2 = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("Without stratify:")
print(y_train1.value_counts(normalize=True))
print("\nWith stratify:")
print(y_train2.value_counts(normalize=True))


In [ ]:
#Observation:
#stratify=y ensures both train and test sets reflect the same class proportions as your full dataset (70–30).
#Without it, your training set may become more imbalanced — leading to biased learning.

# 2.SMOTE

In [ ]:
#After splitting, you may find your training data is still imbalanced (e.g., 500 zeros vs 90 ones).
#SMOTE (Synthetic Minority Over-sampling Technique)
# solves this by creating new minority samples through interpolation — not by simple duplication.

# Demo

In [ ]:
print("Before SMOTE:")
print(y_train.value_counts())


In [ ]:
from imblearn.over_sampling import SMOTE

# Try reducing k_neighbors to 1 or 2
smote = SMOTE(random_state=42, k_neighbors=1)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:\n", y_train_res.value_counts())


In [ ]:
#Observation:
#SMOTE balances the classes by synthesizing new minority-class samples.
#It improves model learning on rare outcomes without biasing the test set (since test data remains untouched).